# 📖 Notebook 7: GitOps with ArgoCD — Deploying from Git

Welcome! In this notebook, you will learn a deployment style called **GitOps**. The big idea is simple: instead of clicking buttons or running lots of manual commands every time you want to deploy, you store the desired Kubernetes state in Git and let a tool keep the cluster matched to that Git history.

We will use the sample microservices in the `k8s-lab` namespace:
- `api-gateway` on port `8000`
- `user-service` on port `8001`
- `order-service` on port `8002`

Think of Git as the **blueprint drawer** and ArgoCD as the **worker that keeps rebuilding the real system to match the blueprint**.

> **Prerequisites: Notebooks 01–03.** The self-heal demo deletes and expects ArgoCD to
> restore a deployment in `k8s-lab`.
>
> **Resources**: ArgoCD's default install is 7 workloads (application-controller, repo-server,
> server, redis, dex, applicationset-controller, notifications-controller). If Notebook 05's
> monitoring stack is still installed and your laptop is tight, `helm uninstall prometheus -n
> monitoring` first.

In [ ]:
# ── Preflight ────────────────────────────────────────────────────────────
# Every later cell shells out to these tools. Without this check a missing
# binary fails silently inside a `!` magic and you only see a confusing
# downstream error (e.g. FileNotFoundError from %%writefile) instead of
# "helm is not installed". Run this first.
import shutil
import subprocess

REQUIRED = ['kubectl', 'git']
INSTALL_HINTS = {
    'kubectl': 'https://kubernetes.io/docs/tasks/tools/  (or `brew install kubectl`)',
    'git': 'https://git-scm.com/downloads  (or `brew install git`)',
}

missing = [b for b in REQUIRED if shutil.which(b) is None]
if missing:
    hint = '\n'.join(f'  - {b}: {INSTALL_HINTS[b]}' for b in missing)
    raise RuntimeError(
        f"Missing required CLI tool(s): {', '.join(missing)}\n"
        f"Install them, then re-run this cell:\n{hint}"
    )

# A reachable cluster is required too -- `kubectl` alone is not enough.
probe = subprocess.run(
    ['kubectl', 'cluster-info'], capture_output=True, text=True
)
if probe.returncode != 0:
    raise RuntimeError(
        'No reachable Kubernetes cluster. Start the one from notebook 1:\n'
        '  minikube start --cpus=4 --memory=6144 --driver=docker\n'
        f'kubectl said: {probe.stderr.strip()[:300]}'
    )

print('Preflight OK:', ', '.join(REQUIRED), '+ cluster reachable')

In [ ]:
# ── Helpers ──────────────────────────────────────────────────────────────
import json
import os
import subprocess
import time

NS = "k8s-lab"


def kget(*args, ns=None):
    cmd = ["kubectl", "get", *args, "-o", "json"] + (["-n", ns] if ns else [])
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("kubectl failed: " + r.stderr.strip()[:400])
    return json.loads(r.stdout)


def wait_until(predicate, timeout=150, interval=5, what="condition"):
    deadline = time.time() + timeout
    while time.time() < deadline:
        value = predicate()
        if value:
            return value
        time.sleep(interval)
    raise AssertionError(f"timed out after {timeout}s waiting for {what}")


def poll(predicate, timeout, interval=10):
    """Same as wait_until, but returns None instead of raising.

    Used where the wait is longer than one notebook cell should block for and is
    therefore continued in the following cell."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        value = predicate()
        if value:
            return value
        time.sleep(interval)
    return None


ARGOCD_WORKLOADS = {
    "deployment": ("argocd-repo-server", "argocd-server",
                   "argocd-applicationset-controller"),
    "statefulset": ("argocd-application-controller",),
}


def argocd_up():
    counts = {}
    for kind, names in ARGOCD_WORKLOADS.items():
        for name in names:
            r = subprocess.run(["kubectl", "get", kind, name, "-n", "argocd",
                                "-o", "json"], capture_output=True, text=True)
            counts[name] = (json.loads(r.stdout)["status"].get("readyReplicas", 0)
                            if r.returncode == 0 else 0)
    print("  ready:", counts)
    return counts if all(v >= 1 for v in counts.values()) else None


# Every git command in this notebook runs with an explicit `cwd=`, never after a
# `cd` that might have failed. That is not fussiness: `cd nonexistent-dir` in a
# shell cell is a WARNING, not an error, so the `git commit` and `git revert`
# further down would then run in whatever directory the notebook happens to live
# in -- which is inside a real git repository. Committing to, and then reverting
# a commit in, the reader's own repo is not a hypothetical failure mode.
def git(*args, cwd, check=True):
    """Run git in `cwd`. Raises if `cwd` is not a git working tree."""
    if not os.path.isdir(os.path.join(cwd, ".git")):
        raise RuntimeError(
            f"{cwd!r} is not a git repository -- refusing to run `git {args[0]}` "
            "somewhere it might hit a repository that is not this lab's scratch copy."
        )
    env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
    r = subprocess.run(["git", *args], cwd=cwd, env=env,
                       capture_output=True, text=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"git {' '.join(args)} failed:\n{r.stderr[-600:]}")
    return r


print("helpers ready")

## Learning Objectives

By the end of this notebook, you will be able to:

- explain the main GitOps idea in beginner-friendly words
- describe why Git becomes the single source of truth
- install ArgoCD into a Kubernetes cluster
- create an ArgoCD Application for the `k8s-lab` manifests
- observe sync, auto-sync, and self-healing behavior
- understand how Git commits and Git revert can act like deploy and rollback buttons

## 🛠️ Setup

Before starting:

1. Make sure your Kubernetes cluster is running.
2. Make sure the `k8s-lab` namespace and sample services already exist.
3. Make sure Git is installed on your machine.
4. Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → 'Reload Window'.

This notebook uses shell commands inside code cells, so you can follow the flow step by step.

In [ ]:
!kubectl cluster-info
!kubectl get ns
!kubectl get deployments -n k8s-lab
!git --version

## What is GitOps?

GitOps means **Git describes what the cluster should look like**. A tool such as ArgoCD keeps checking Git and comparing it with the real cluster. If the cluster drifts away from the files in Git, ArgoCD can bring it back.

A good beginner mental model is:

- **Git** = your saved plan
- **Kubernetes** = the real running system
- **ArgoCD** = the careful robot inspector

### 🧪 Practical Exercise
Look at the existing manifests in this lab and say out loud what they describe: namespaces, deployments, services, and other cluster objects. The goal is to notice that GitOps works best when your desired state is already written down as files.

In [ ]:
# This notebook runs from `notebooks/`, so the shared manifests are one level up.
!ls -1 ../manifests
!kubectl get all -n k8s-lab

## 📦 Git Push → ArgoCD → Kubernetes

```text
+-------------------+        +-------------------+        +------------------------+
|   Your Git Repo   | -----> |      ArgoCD       | -----> |   Kubernetes Cluster   |
| manifests/*.yaml  |        | watches Git state |        | runs real workloads    |
+-------------------+        +-------------------+        +------------------------+
         ^                              |                               |
         |                              v                               v
         +---------------------- Git is the source ---------------------+
```

ArgoCD keeps asking one question: **"Does the cluster still match Git?"**

## Traditional Deploy vs GitOps

| Topic | Traditional Deploy | GitOps |
|---|---|---|
| Source of truth | Terminal history, wiki pages, memory | Git repository |
| Change process | Humans run commands by hand | Humans change Git, ArgoCD applies it |
| Audit trail | Harder to reconstruct | Built into Git commits |
| Rollback | Manual and sometimes stressful | Revert a commit and sync again |
| Drift recovery | Someone must notice and fix it | ArgoCD can self-heal |
| Team collaboration | Easy to make different changes in different ways | Everyone works through the same repo |

### 🧪 Practical Exercise
Pick one row from the table and explain why it matters to a small team. For example, why is an audit trail useful when several people deploy the same app?

## 1) Install ArgoCD

First, we install ArgoCD into its own namespace. This adds controllers, APIs, and a web UI.

### 🧪 Practical Exercise
Before running the install, guess what will happen after we create the `argocd` namespace: will you see more pods, more services, or both? Then run the cell and check your answer.

In [ ]:
# create -> apply, so re-running this notebook does not fail on AlreadyExists.
!kubectl create namespace argocd --dry-run=client -o yaml | kubectl apply -f -
!kubectl apply -n argocd -f https://raw.githubusercontent.com/argoproj/argo-cd/stable/manifests/install.yaml

print()
# The manifest creates ~7 workloads. Applying is instant; pulling their images is
# not, which is why waiting for them is a separate cell below.
!kubectl get deploy,statefulset -n argocd

In [ ]:
# ArgoCD's manifest creates seven workloads and pulls several images. On a fresh
# cluster that is minutes of downloading -- longer than a notebook runner lets one
# cell block -- so the wait is a bounded poll, continued in the next cell.
print("waiting for the ArgoCD control plane:")
state = argocd_up() or poll(argocd_up, timeout=160)
print("all up" if state else "still pulling -- the next cell keeps waiting")

In [ ]:
state = argocd_up() or poll(argocd_up, timeout=170)
!kubectl get pods -n argocd

# The self-heal demo below is driven by the application-controller and the
# repo-server. If either is not up, the demo silently "works" by never
# reconciling at all, so check before relying on them.
assert state, (
    "ArgoCD never became Ready. `kubectl get pods -n argocd` and `kubectl describe` "
    "will say why -- on a fresh cluster it is usually still pulling images, in which "
    "case re-running this cell continues the wait."
)
print("\n✅ ArgoCD control plane is up")

## 2) Access the ArgoCD UI

The ArgoCD server runs inside the cluster. To open it in your browser from your laptop, we use **port-forwarding**. That creates a temporary tunnel from your machine to the ArgoCD service.

Then we fetch the initial admin password from a Kubernetes secret.

### 🧪 Practical Exercise
After starting the port-forward, open `https://localhost:8080` in your browser. Log in with username `admin` and the password from the next command. Notice how the UI starts empty before any Application is created.

In [ ]:
# `!kubectl port-forward ... &` does NOT work in a notebook: the `!` magic runs the
# command in a subshell that is torn down as soon as the cell finishes, taking the
# background job with it. Use subprocess.Popen, which is owned by the kernel and
# survives until you terminate it (or the kernel dies).
import base64
import subprocess
import time

argocd_pf = subprocess.Popen(
    ["kubectl", "port-forward", "svc/argocd-server", "-n", "argocd", "8080:443"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
time.sleep(3)

# Decode the bootstrap password in Python -- `base64 -d` vs `-D` differs between
# GNU and BSD/macOS, and this way there is nothing to get wrong.
raw = subprocess.run(
    ["kubectl", "-n", "argocd", "get", "secret", "argocd-initial-admin-secret",
     "-o", "jsonpath={.data.password}"],
    capture_output=True, text=True,
).stdout
password = base64.b64decode(raw).decode() if raw else "(secret not found -- already rotated?)"

print("ArgoCD UI: https://localhost:8080")
print("  (self-signed cert -- your browser will warn; proceed anyway)")
print("  username: admin")
print(f"  password: {password}")
print()
print("Optional CLI: brew install argocd   (or see the official install docs)")

## 3) Create a local Git repo for GitOps

Now we create a small Git repository that contains the Kubernetes manifests. This helps you see the Git side of GitOps clearly. To keep the lab self-contained, we will place it in `./gitops-workdir/k8s-lab-gitops`.

> Important beginner note: ArgoCD cannot watch a random folder on your laptop directly. It watches a Git repository that **it can reach**. So we will create a local repo first because it is easy to understand, and then point ArgoCD at a remote Git URL that you push to.

### 🧪 Practical Exercise
After the commit finishes, run `git log --oneline` mentally or in the terminal and notice that your desired cluster state now has a version number: a commit hash. That is one of the biggest GitOps wins.

In [ ]:
# The notebook runs from `notebooks/`, so the shared manifests are at ../manifests.
# (Copying `manifests` -- as if we were one directory up -- fails, and in a chained
# shell command that failure then cascades into git commands running in the wrong
# directory. Hence the explicit paths and the checks below.)
import shutil

REPO_DIR = os.path.abspath("./gitops-workdir/k8s-lab-gitops")
SOURCE_MANIFESTS = os.path.abspath("../manifests")
assert os.path.isdir(SOURCE_MANIFESTS), f"cannot find {SOURCE_MANIFESTS}"

shutil.rmtree(os.path.dirname(REPO_DIR), ignore_errors=True)
os.makedirs(REPO_DIR, exist_ok=True)
shutil.copytree(SOURCE_MANIFESTS, os.path.join(REPO_DIR, "manifests"))

subprocess.run(["git", "init", "-b", "main"], cwd=REPO_DIR, check=True,
               capture_output=True, text=True)
git("config", "user.name", "K8s Lab", cwd=REPO_DIR)
git("config", "user.email", "lab@example.com", cwd=REPO_DIR)
git("add", ".", cwd=REPO_DIR)
git("commit", "-m", "Initial GitOps manifests", cwd=REPO_DIR)

print(git("log", "--oneline", "-n", "3", cwd=REPO_DIR).stdout)

# The desired cluster state now has a version number.
head = git("rev-parse", "--short", "HEAD", cwd=REPO_DIR).stdout.strip()
tracked = git("ls-files", cwd=REPO_DIR).stdout.split()
assert any(f.startswith("manifests/") for f in tracked), \
    f"the manifests were not committed: {tracked}"
print(f"✅ scratch repo at {REPO_DIR}, HEAD={head}, {len(tracked)} files tracked")

## 4) Create an ArgoCD Application

An **Application** is the object that tells ArgoCD:

- which repo to watch (`spec.source.repoURL`)
- which folder inside that repo to read (`spec.source.path`)
- which cluster and namespace to deploy into (`spec.destination`)
- how much to automate (`spec.syncPolicy`)

### The catch with a local repo

ArgoCD's repo-server runs **inside the cluster**. It clones over the network. It cannot
see `./gitops-workdir` on your laptop, and it cannot see `http://localhost` either —
`localhost` inside that pod is the pod itself. So a Git repo that only exists on your
disk is not something ArgoCD can sync from, no matter what path you give it.

That leaves two honest options, and we will do both:

- **A — a real end-to-end sync**, right now, from a public repo ArgoCD can definitely
  reach: `argoproj/argocd-example-apps`. This is the one that actually demonstrates
  auto-sync and self-healing.
- **B — the template for your own repo**, which you can use after pushing
  `./gitops-workdir/k8s-lab-gitops` to GitHub.

### ⚠️ Read `syncPolicy` before you copy it anywhere

```yaml
syncPolicy:
  automated:
    prune: true      # delete cluster objects that are no longer in Git
    selfHeal: true   # revert any manual change to the cluster
```

- `prune: true` means **ArgoCD will delete resources**. If someone moves a file out of the
  watched path, the corresponding live object is removed from the cluster. That is the
  intended behaviour and it is also how people accidentally delete production. Start with
  `prune: false` on a real system until you trust the repo layout.
- `selfHeal: true` means `kubectl edit` on a managed object is undone within minutes.
  Excellent for drift, infuriating during an incident — know how to pause it
  (`argocd app set <app> --sync-policy none`).
- **Never put the Application manifest inside the path the Application watches.** Ours
  would tell ArgoCD to create an `Application` object in the `k8s-lab` namespace, where
  the ArgoCD controller does not look for them. Keep it beside the notebook instead.

### 🧪 Practical Exercise
Read the YAML before applying it and answer: (1) what namespace will ArgoCD deploy into?
(2) which two automation features are turned on?

In [ ]:
%%writefile ./argocd-app-guestbook.yaml
# Option A: a public repo the in-cluster repo-server can definitely clone.
# This is the one that actually syncs, so the demos below have something to work on.
apiVersion: argoproj.io/v1alpha1
kind: Application
metadata:
  name: guestbook
  namespace: argocd
spec:
  project: default
  source:
    repoURL: https://github.com/argoproj/argocd-example-apps.git
    targetRevision: HEAD
    path: guestbook
  destination:
    server: https://kubernetes.default.svc
    namespace: gitops-demo
  syncPolicy:
    automated:
      prune: true
      selfHeal: true
    syncOptions:
      - CreateNamespace=true

In [ ]:
!kubectl apply -f ./argocd-app-guestbook.yaml


def app_status():
    app = kget("application", "guestbook", ns="argocd")
    return app.get("status", {})


# Give the controller time to clone and sync. Poll rather than `kubectl wait
# --timeout=300s`, so this cell cannot outlive a notebook-runner's cell budget.
status = wait_until(
    lambda: app_status() if app_status().get("sync", {}).get("status") == "Synced" else None,
    timeout=170, interval=5, what="the guestbook Application to reach Synced")

!kubectl get application -n argocd
!kubectl get all -n gitops-demo

# "Synced" means the cluster matches Git; "Healthy" means the workloads it
# created actually came up. They are different questions and both matter.
assert status["sync"]["status"] == "Synced", status["sync"]
health = wait_until(
    lambda: app_status()["health"]["status"]
    if app_status().get("health", {}).get("status") == "Healthy" else None,
    timeout=150, interval=5, what="the guestbook Application to become Healthy")

dep = kget("deployment", "guestbook-ui", ns="gitops-demo")
assert dep["status"].get("readyReplicas", 0) >= 1, \
    "ArgoCD reported Healthy but guestbook-ui has no ready replicas"
print(f"\n✅ ArgoCD cloned a repo and deployed it: sync={status['sync']['status']}, "
      f"health={health}, and it did so with no `kubectl apply` from us")

`SYNC STATUS: Synced` and `HEALTH STATUS: Healthy` mean ArgoCD cloned the repo, rendered
`guestbook/`, applied it, and confirmed the workloads came up — with no `kubectl apply`
from you.

Below is the template for **your own** repo. Push `./gitops-workdir/k8s-lab-gitops` to
GitHub first, replace `YOUR_USERNAME`, then apply it. Note that it points at
`path: manifests` in *your* repo, and that it deploys into `k8s-lab`.

In [ ]:
%%writefile ./argocd-app-k8s-lab.yaml
# Option B: your own repo. Replace YOUR_USERNAME and push the local repo first,
# otherwise ArgoCD will report `ComparisonError: repository not found`.
apiVersion: argoproj.io/v1alpha1
kind: Application
metadata:
  name: k8s-lab-app
  namespace: argocd
spec:
  project: default
  source:
    repoURL: https://github.com/YOUR_USERNAME/k8s-lab-gitops.git
    targetRevision: main
    path: manifests
    directory:
      recurse: true
  destination:
    server: https://kubernetes.default.svc
    namespace: k8s-lab
  syncPolicy:
    automated:
      prune: true
      selfHeal: true
    syncOptions:
      - CreateNamespace=true

In [ ]:
# Only apply Option B once you have pushed your repo and edited YOUR_USERNAME above.
# It is left commented out so re-running this notebook does not create a permanently
# broken Application.
# !kubectl apply -f ./argocd-app-k8s-lab.yaml
!cat ./argocd-app-k8s-lab.yaml

## 5) Observe sync and self-healing

Once the Application exists, ArgoCD compares Git with the cluster every ~3 minutes (and
immediately when you ask it to). If something in the cluster no longer matches Git,
`selfHeal: true` makes ArgoCD put it back.

We test this on the **guestbook** app, because that one is really synced from a repo
ArgoCD can read.

### 🧪 Practical Exercise
Delete the guestbook deployment by hand and predict what ArgoCD will do. Ignore it?
Complain in the UI only? Recreate it?

In [ ]:
!kubectl get application guestbook -n argocd

# Record the object's UID first. A UID is assigned at creation and never reused,
# so "same name, different UID" is proof the object was really destroyed and
# recreated -- which is the only way to tell self-heal apart from "nothing
# happened". ArgoCD watches its managed objects, so the replacement can arrive
# in well under a second: checking for absence is a race you will lose.
before = kget("deployment", "guestbook-ui", ns="gitops-demo")["metadata"]["uid"]
print("\nguestbook-ui UID before:", before)

print("\n--- manual drift: delete a managed object ---")
!kubectl delete deployment guestbook-ui -n gitops-demo

print("\n--- waiting for the self-heal loop ---")
# Auto-sync reconciles roughly every 3 minutes on a timer, but a delete of a
# managed object is also watched, so in practice this is seconds.


def recreated():
    r = subprocess.run(["kubectl", "get", "deployment", "guestbook-ui",
                        "-n", "gitops-demo", "-o", "json"],
                       capture_output=True, text=True)
    if r.returncode != 0:
        return None
    dep = json.loads(r.stdout)
    return dep if dep["metadata"]["uid"] != before else None


dep = wait_until(recreated, timeout=170, interval=2,
                 what="ArgoCD to recreate the deployment it manages")
print("guestbook-ui UID after: ", dep["metadata"]["uid"])
print("✅ ArgoCD recreated it -- that is selfHeal.")
!kubectl get deployment -n gitops-demo

# Recreated is not the same as recreated correctly, so check it came back with
# the spec from Git rather than as an empty shell.
assert dep["spec"]["replicas"] >= 1 and dep["spec"]["template"]["spec"]["containers"], \
    f"guestbook-ui came back malformed: {dep['spec']}"
synced = wait_until(
    lambda: kget("application", "guestbook", ns="argocd")["status"]["sync"]["status"] == "Synced",
    timeout=150, interval=5, what="the Application to return to Synced")
print("\n✅ drift detected and reverted; Application back to Synced")

## 6) Make a change in Git and watch auto-sync

A very common GitOps flow is:

1. edit the manifest
2. commit the change
3. push the change
4. let ArgoCD notice and sync it

Below, we change the `api-gateway` replica count from `2` to `3`.

### 🧪 Practical Exercise
Before you run the cell, guess where you will see the new desired state first: Git history, ArgoCD UI, or the Kubernetes Deployment object. Then run the commands and compare the order.

In [ ]:
# ArgoCD's repo-server clones from inside the cluster, so the local scratch repo
# is not something it can sync from. Set REMOTE to a repo you own and have pushed
# to, and this section becomes a real end-to-end GitOps change.
#
# GIT_TERMINAL_PROMPT=0 matters here: without it, pushing to a repo that does not
# exist makes git block forever waiting for a username on a terminal the notebook
# does not have, and the cell simply hangs. `git()` sets it for every call.
REMOTE = "https://github.com/YOUR_USERNAME/k8s-lab-gitops.git"
print("Set REMOTE to your own repo URL to make this section do something real.")
print("REMOTE =", REMOTE)

In [ ]:
from pathlib import Path

# 1. Point at the remote (idempotent). `check=False` because "no such remote" on
#    the first run is expected, not a failure.
git("remote", "remove", "origin", cwd=REPO_DIR, check=False)
git("remote", "add", "origin", REMOTE, cwd=REPO_DIR)

# 2. Make the change locally. This half works whether or not you have a remote.
target = Path(REPO_DIR) / "manifests" / "deployment.yaml"
before = target.read_text()
if "replicas: 2" in before:
    target.write_text(before.replace("replicas: 2", "replicas: 3", 1))
    print("patched api-gateway to replicas: 3")
else:
    print("already patched -- nothing to do")
assert "replicas: 3" in target.read_text(), "the replica count was not changed"

git("add", "manifests/deployment.yaml", cwd=REPO_DIR)
commit = git("commit", "-m", "Scale api-gateway to 3 replicas", cwd=REPO_DIR, check=False)
print(commit.stdout.strip() or commit.stderr.strip())
print(git("log", "--oneline", "-n", "3", cwd=REPO_DIR).stdout)

# The desired state is now versioned: the change exists as a commit before it
# exists anywhere in the cluster.
log = git("log", "-1", "--pretty=%s", cwd=REPO_DIR).stdout.strip()
assert log == "Scale api-gateway to 3 replicas", f"unexpected HEAD commit: {log!r}"
assert git("status", "--porcelain", cwd=REPO_DIR).stdout.strip() == "", \
    "the change should be committed, not left in the working tree"

# 3. Push. Fails harmlessly if YOUR_USERNAME was never replaced.
push = git("push", "-u", "origin", "main", cwd=REPO_DIR, check=False)
if push.returncode == 0:
    print("pushed to", REMOTE)
else:
    print(">>> push skipped: set REMOTE to a repo you own")
    print("   ", push.stderr.strip().splitlines()[0][:120] if push.stderr.strip() else "")

Note the ordering the exercise asked about: the change exists in **Git history first**,
then in the ArgoCD UI as `OutOfSync`, and only then in the live Deployment object. In a
`kubectl apply` workflow the cluster changes first and Git may never catch up at all —
that gap is exactly what GitOps closes.

## 7) Roll back with Git revert

Rollback in GitOps is beautifully simple: if a bad change entered the repo, undo the commit and let the system converge back to the older good state.

### 🧪 Practical Exercise
Run the revert and then explain why this rollback is easier to review than running random emergency commands straight in the cluster.

In [ ]:
# `git revert` creates a NEW commit that undoes the previous one. It does not
# rewrite history, which is why it is the safe rollback in a shared repo --
# `git reset --hard` + force-push would break every other clone.
#
# Note this runs against REPO_DIR explicitly. A `cd` into a directory that does
# not exist is only a warning in a shell cell, and `git revert HEAD` in whatever
# directory you land in instead will happily revert a commit in a completely
# different repository. `git()` refuses to run anywhere that is not a git tree.
head_before = git("rev-parse", "HEAD", cwd=REPO_DIR).stdout.strip()

rev = git("revert", "--no-edit", "HEAD", cwd=REPO_DIR, check=False)
print(rev.stdout.strip() or rev.stderr.strip())
print(git("log", "--oneline", "-n", "3", cwd=REPO_DIR).stdout)

push = git("push", cwd=REPO_DIR, check=False)
print("pushed" if push.returncode == 0 else ">>> push skipped: no reachable remote")

# Rollback in GitOps = the file is back to its old contents, and the history
# still shows both the change and its undo.
from pathlib import Path

target = Path(REPO_DIR) / "manifests" / "deployment.yaml"
assert "replicas: 3" not in target.read_text(), \
    "the revert did not restore the manifest"
subjects = git("log", "-3", "--pretty=%s", cwd=REPO_DIR).stdout.split("\n")
assert subjects[0].startswith('Revert "Scale api-gateway to 3 replicas"'), \
    f"expected a revert commit on top, got {subjects[:3]}"
assert git("rev-parse", "HEAD", cwd=REPO_DIR).stdout.strip() != head_before, \
    "revert should add a commit, not move the branch backwards"
print("\n✅ rolled back by adding a commit -- the change and its undo are both auditable")

## 🧹 Clean Up

When you finish exploring, remove ArgoCD from the cluster so the lab returns to a simpler state.

### 🧪 Practical Exercise
After deleting the namespace, run `kubectl get ns` and verify that `argocd` disappears. This helps reinforce that most Kubernetes tools are just resources living in namespaces.

In [ ]:
# Delete the Applications first. With `prune: true`, deleting the argocd namespace
# out from under a running controller can leave the guestbook objects orphaned.
!kubectl delete -f ./argocd-app-guestbook.yaml --ignore-not-found
!kubectl delete namespace gitops-demo --ignore-not-found

!kubectl delete namespace argocd --ignore-not-found

# Stop the port-forward we started with subprocess.Popen.
try:
    argocd_pf.terminate()
except NameError:
    pass

!rm -f ./argocd-app-guestbook.yaml ./argocd-app-k8s-lab.yaml

# Only ever remove the scratch tree we created, by the absolute path we built it at.
shutil.rmtree(os.path.dirname(REPO_DIR), ignore_errors=True)
assert not os.path.exists(REPO_DIR)

!kubectl get ns

for _ns in ("argocd", "gitops-demo"):
    gone = subprocess.run(["kubectl", "get", "namespace", _ns],
                          capture_output=True).returncode != 0
    assert gone, f"namespace {_ns} is still present -- notebook 08 needs the memory"
print("\n✅ ArgoCD removed; the cluster is back to the state notebook 08 expects")

## 🎓 What You Learned

Nice work. In this notebook, you learned that:

- GitOps uses Git as the single source of truth
- ArgoCD watches Git and compares it to the live cluster
- auto-sync lets changes flow from Git to Kubernetes automatically
- self-heal lets ArgoCD repair drift when someone changes the cluster manually
- Git commits and Git revert create a clean, reviewable deployment history
- ArgoCD's repo-server clones **from inside the cluster**, so it cannot reach a repo
  that only exists on your laptop, nor anything on your `localhost`
- `prune: true` deletes live objects that leave Git, and `selfHeal: true` reverts manual
  `kubectl edit`s — both are the point of GitOps and both need to be understood before
  you enable them on something that matters
- Never store an Application manifest inside the path that Application watches

If you can explain the difference between **changing the cluster directly** and **changing Git first**, you now understand the heart of GitOps.